In [27]:
#download needed libraries,
%pip install kagglehub numpy pandas scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\AVENg\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [28]:
import pandas as pd
import numpy as np

# try to get the file or crash out if missing
try:
    df = pd.read_csv('.\DATA\Airbnb_Open_Data.csv')
    print("row count:", len(df))
except:
    print("fix path to Airbnb_Open_Data.csv")

# lower case cols and add underscores
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# clean up the messy price columns 
for c in ['price', 'service_fee']:
    if c in df.columns:
        df[c] = df[c].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
        df[c] = pd.to_numeric(df[c], errors='coerce')

print("\ncols ready:")
print(df[['price', 'service_fee']].dtypes)
display(df.head())

C:\Users\AVENg\AppData\Local\Temp\ipykernel_19584\2515993062.py:6: DtypeWarning: Columns (0: license) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('.\DATA\Airbnb_Open_Data.csv')


row count: 102599

cols ready:
price          float64
service_fee    float64
dtype: object


,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,...,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,193.0,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,28.0,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,124.0,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,74.0,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,41.0,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN


In [29]:
# 1. fix neighborhood typos and blanks
if 'neighbourhood_group' in df.columns:
    df['neighbourhood_group'] = df['neighbourhood_group'].fillna('Unknown').replace({'brookln': 'Brooklyn', 'manhatan': 'Manhattan'})

# fill policy and room types with mode
df['cancellation_policy'] = df['cancellation_policy'].fillna(df['cancellation_policy'].mode()[0])
df['room_type'] = df['room_type'].fillna(df['room_type'].mode()[0])

# 2. replace all numerical nan types with their column median
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# 3. fill any leftover text columns with 'Unknown'
text_cols = df.select_dtypes(include=['object']).columns
df[text_cols] = df[text_cols].fillna('Unknown')

print("nulls left:")
print(df[['price', 'service_fee', 'construction_year', 'neighbourhood_group']].isnull().sum())
display(df.head())

nulls left:
price                  0
service_fee            0
construction_year      0
neighbourhood_group    0
dtype: int64


C:\Users\AVENg\AppData\Local\Temp\ipykernel_19584\3075418412.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include=['object']).columns


,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,...,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,193.0,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,Unknown
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,28.0,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,Unknown
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,Unknown,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,124.0,3.0,0.0,Unknown,0.74,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",Unknown
3,1002755,Unknown,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,74.0,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,Unknown,Unknown
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,41.0,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",Unknown


In [30]:
#remove rows with negative or zer0 values
bad_rows = df[df['price'] <= 0].index
df = df.drop(index=bad_rows).reset_index(drop=True)
print(f"dropped {len(bad_rows)} zero-price rows")

#fee ratio
df['fee_to_price_ratio'] = df['service_fee'] / df['price']
print(df['fee_to_price_ratio'].describe())
display(df.head())

dropped 0 zero-price rows
count    102599.000000
mean          0.200246
std           0.018247
min           0.016026
25%           0.199605
50%           0.200000
75%           0.200391
max           2.500000
Name: fee_to_price_ratio, dtype: float64


,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,...,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,license,fee_to_price_ratio
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,Unknown,0.199793
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,Unknown,0.197183
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,Unknown,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,3.0,0.0,Unknown,0.74,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",Unknown,0.200000
3,1002755,Unknown,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,Unknown,Unknown,0.201087
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",Unknown,0.200980


In [31]:
# custom min max scaling for year built
y_min = df['construction_year'].min()
y_max = df['construction_year'].max()

if (y_max - y_min) != 0:
    df['construction_year_scaled'] = (df['construction_year'] - y_min) / (y_max - y_min)
else:
    df['construction_year_scaled'] = 0.0

print("scale check:")
print("min:", df['construction_year_scaled'].min(), "max:", df['construction_year_scaled'].max())
display(df.head())

scale check:
min: 0.0 max: 1.0


,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,...,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,license,fee_to_price_ratio,construction_year_scaled
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,Unknown,0.199793,0.894737
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,Unknown,0.197183,0.210526
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,Unknown,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,0.0,Unknown,0.74,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",Unknown,0.200000,0.105263
3,1002755,Unknown,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,270.0,7/5/2019,4.64,4.0,1.0,322.0,Unknown,Unknown,0.201087,0.105263
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",Unknown,0.200980,0.315789


In [32]:
# print original price skew
print("skew value:", df['price'].skew())

# mean and std calculation
avg_p = df['price'].mean()
std_p = df['price'].std()

# apply z-score standardizing
if std_p != 0:
    df['price_normalized'] = (df['price'] - avg_p) / std_p
else:
    df['price_normalized'] = 0.0

print(f"done -> mean: {df['price_normalized'].mean():.4f} | std: {df['price_normalized'].std():.4f}")
display(df.head())

skew value: 0.0013664423371729767
done -> mean: 0.0000 | std: 1.0000


,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,...,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,license,fee_to_price_ratio,construction_year_scaled,price_normalized
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,Unknown,0.199793,0.894737,1.028488
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,Unknown,0.197183,0.210526,-1.458892
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,Unknown,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,Unknown,0.74,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",Unknown,0.200000,0.105263,-0.015970
3,1002755,Unknown,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,7/5/2019,4.64,4.0,1.0,322.0,Unknown,Unknown,0.201087,0.105263,-0.776674
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",Unknown,0.200980,0.315789,-1.271735


In [33]:
# use pd.get_dummies to convert room_type and cancellation_policy into binary 1s and 0s

In [34]:
# check instant_bookable counts and downsample the majority class using sklearn resample

In [35]:
# convert neighbourhood_group strings into simple integer codes using .cat.codes

In [36]:
# drop unneeded text metadata columns and export the clean dataframe to a new csv file

VISUALIZE DATA:


In [ ]:
#matplotlib stuff